In [1]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score, RandomizedSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder, StandardScaler, MinMaxScaler
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, classification_report, confusion_matrix,precision_recall_curve
)

from sklearn.pipeline import Pipeline

import optuna

In [2]:
df = pd.read_csv('../data/cleaned_loan_data')
df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,home_ownership,annual_inc,verification_status,...,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim,target
0,5000,5000,4975.0,36 months,10.65,162.87,B,RENT,24000.0,Verified,...,13648,83.7,9.0,f,0.0,0.0,0.0,81539.0,22800.0,0
1,2500,2500,2500.0,60 months,15.27,59.83,C,RENT,30000.0,Source Verified,...,1687,9.4,4.0,f,0.0,0.0,0.0,81539.0,22800.0,1
2,2400,2400,2400.0,36 months,15.96,84.33,C,RENT,12252.0,Not Verified,...,2956,98.5,10.0,f,0.0,0.0,0.0,81539.0,22800.0,0
3,10000,10000,10000.0,36 months,13.49,339.31,C,RENT,49200.0,Source Verified,...,5598,21.0,37.0,f,0.0,0.0,0.0,81539.0,22800.0,0
4,3000,3000,3000.0,60 months,12.69,67.79,B,RENT,80000.0,Source Verified,...,27783,53.9,38.0,f,0.0,0.0,0.0,81539.0,22800.0,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 466285 entries, 0 to 466284
Data columns (total 26 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   loan_amnt                   466285 non-null  int64  
 1   funded_amnt                 466285 non-null  int64  
 2   funded_amnt_inv             466285 non-null  float64
 3   term                        466285 non-null  str    
 4   int_rate                    466285 non-null  float64
 5   installment                 466285 non-null  float64
 6   grade                       466285 non-null  str    
 7   home_ownership              466285 non-null  str    
 8   annual_inc                  466285 non-null  float64
 9   verification_status         466285 non-null  str    
 10  dti                         466285 non-null  float64
 11  delinq_2yrs                 466285 non-null  float64
 12  inq_last_6mths              466285 non-null  float64
 13  mths_since_last_delinq   

In [4]:
df['term'] = df['term'].astype('str')
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)

df['home_ownership'] = df['home_ownership'].replace(
    {'NONE': 'OTHER', 'ANY': 'OTHER'}
)


In [5]:
nominal_features   = ['home_ownership', 'verification_status', 'initial_list_status']
ordinal_features   = ['grade']
ordinal_categories = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]

num_features = [c for c in df.select_dtypes(include=['int64', 'float64']).columns
                if c != 'target'
                and c not in ordinal_features
                and c not in nominal_features]

target = 'target'

X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape,y_train.shape,X_test.shape,y_test.shape)

(373028, 25) (373028,) (93257, 25) (93257,)


In [6]:
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos:.2f}")  # sekitar 7.4

models = {
    'Logistic Regression': LogisticRegression(
        random_state=42, max_iter=1000, n_jobs=-1,
        class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42, n_jobs=-1, n_estimators=100,
        class_weight='balanced'
    ),
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(class_weight='balanced'),
        random_state=42
    ),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(
        random_state=42, n_jobs=-1,
        eval_metric='logloss',
        scale_pos_weight=scale_pos  
    ),
}

scale_pos_weight: 7.43


In [7]:
numeric_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Ordinal
ordinal_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Nominal
nominal_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,  num_features),
    ('ord', ordinal_pipeline,  ordinal_features),
    ('nom', nominal_pipeline,  nominal_features),
], remainder='drop')

In [13]:
test_results = []

for name, algo in models.items():
    model = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(sampling_strategy=0.5, random_state=42, k_neighbors=5)),
        ('model', algo)
    ])
    model.fit(X_train, y_train)

    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    test_results.append({
        'Model':     name,
        'AUC-ROC':   round(roc_auc_score(y_test, y_pred_proba), 4),
        'F1':        round(f1_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4)
    })

pd.DataFrame(test_results).sort_values('AUC-ROC', ascending=False)

C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


,Model,AUC-ROC,F1,Precision,Recall
0,Logistic Regression,0.6791,0.2871,0.1854,0.6359
5,XGBoost,0.6749,0.2753,0.1699,0.7251
2,Random Forest,0.6628,0.0470,0.2933,0.0256
4,Naive Bayes,0.6549,0.2256,0.1284,0.9266
3,AdaBoost,0.5269,0.1729,0.1577,0.1914
1,Decision Tree,0.5267,0.1728,0.1573,0.1917


In [10]:
RANDOM_STATE = 42
N_ITER= 10
CV_FOLDS= 3
 
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
cv= StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
lr_param_dist = {
    'model__C':        [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
    'model__penalty':  ['l1', 'l2'],
    'model__l1_ratio':  [0.0, 0.25, 0.5, 0.75, 1.0],
    'model__max_iter': [500, 1000, 1500, 2000],
}
 
lr_base = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(sampling_strategy=0.5, random_state=RANDOM_STATE, k_neighbors=5)),
    ('model', LogisticRegression(
        solver='saga',
        class_weight='balanced',
        random_state=RANDOM_STATE,
#         n_jobs=1
    ))
])
 
lr_search = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=lr_param_dist,
    n_iter=N_ITER,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,
    refit=True
)

lr_search.fit(X_train, y_train)
 
print(f"  Best AUC-ROC (CV) : {lr_search.best_score_:.4f}")
print(f"  Best Params:")
for k, v in lr_search.best_params_.items():
    print(f"    {k:<25} : {v}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits


C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of

C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\pikri\ml\Credit-Risk-Modelling\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of